In [ ]:
!pip install torch-geometric-signed-directed

In [ ]:
"""
Retweet Prediction GNN Pipeline
================================
Each sample is a subgraph centered on a (user, tweet) pair.
Node features come from pretrained MagNet embeddings (.pt file).

Graph structure per sample:
  - Central user node
  - 2-hop neighbor user nodes
  - A binary node feature flag: did this neighbor retweet the tweet?

Target: did the central user retweet the tweet? (binary classification)
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data, Dataset
from torch_geometric.loader import DataLoader
from torch_geometric.nn import TransformerConv
from torch_geometric_signed_directed.nn.directed import MagNetConv

In [71]:
import torch
import subprocess

def get_free_memory_per_gpu():
    result = subprocess.run(
        ["nvidia-smi", "--query-gpu=memory.free", "--format=csv,nounits,noheader"],
        capture_output=True, text=True
    )
    return [int(x) for x in result.stdout.strip().split("\n")]

def get_best_gpu():
    free_mem = get_free_memory_per_gpu()
    best_gpu = max(range(len(free_mem)), key=lambda i: free_mem[i])
    print(f"Free memory per GPU (MiB): {free_mem}")
    print(f"Selected GPU {best_gpu} with {free_mem[best_gpu]} MiB free")
    return best_gpu

device = torch.device(f"cuda:{get_best_gpu()}")
# model = model.to(device)

Free memory per GPU (MiB): [171, 24162, 24162, 24162, 2399]
Selected GPU 1 with 24162 MiB free


In [72]:
# ---------------------------------------------------------------------------
# 1. Pretrained Embedding Lookup
# ---------------------------------------------------------------------------

class PretrainedEmbeddingLookup(nn.Module):
    """
    Maps global user IDs to their pretrained MagNet embeddings.
    Embeddings are frozen (not trained).
    """
    def __init__(self, embeddings_path: str, device: str):
        super().__init__()
        # Shape: [num_users, embedding_dim]
        pretrained = torch.load(embeddings_path, weights_only=True, map_location=device)
        # Register as a buffer so it moves with .to(device) but is not a parameter
        self.register_buffer("embeddings", pretrained)
        self.embedding_dim = pretrained.shape[1]

    def forward(self, user_ids: torch.Tensor) -> torch.Tensor:
        """
        Args:
            user_ids: [N] global user IDs (indices into the embedding matrix)
        Returns:
            [N, embedding_dim] pretrained embeddings
        """
        return self.embeddings[user_ids]

In [73]:
# ---------------------------------------------------------------------------
# 2. Dataset
# ---------------------------------------------------------------------------

class RetweetDataset(Dataset):
    """
    Each sample describes the local subgraph around a (central_user, tweet) pair.

    Expected input per sample (raw_samples list):
    {
        "central_user_id":  int,
        "neighbor_ids":     List[int],          # 2-hop neighbors (global user IDs)
        "retweeted_ids":    List[int],           # subset of neighbor_ids that retweeted
        "edge_index":       List[Tuple[int,int]],# edges as LOCAL node index pairs
        "label":            int,                 # 1 = central user retweeted, 0 = did not
    }

    Node ordering convention:
        index 0           → central user
        index 1..N-1      → neighbor users (in the order given by neighbor_ids)
    """

    def __init__(self, raw_samples: list):
        super().__init__()
        self.samples = raw_samples

    def len(self):
        return len(self.samples)

    def get(self, idx):
        s = self.samples[idx]

        # All node IDs in order: central first, then neighbors
        all_ids = [s["central_user_id"]] + list(s["neighbor_ids"])
        num_nodes = len(all_ids)

        user_ids = torch.tensor(all_ids, dtype=torch.long)

        # Binary retweet flag feature for each node
        retweeted_set = set(s["retweeted_ids"])
        retweet_flag = torch.tensor(
            [1.0 if uid in retweeted_set else 0.0 for uid in all_ids],
            dtype=torch.float
        ).unsqueeze(1)  # [N, 1]

        # Edge index (local indices)
        if len(s["edge_index"]) > 0:
            edge_index = torch.tensor(s["edge_index"], dtype=torch.long).t().contiguous()
        else:
            edge_index = torch.zeros((2, 0), dtype=torch.long)

        label = torch.tensor(s["label"], dtype=torch.long)

        return Data(
            user_ids=user_ids,          # [N]   for embedding lookup
            retweet_flag=retweet_flag,  # [N,1] extra structural feature
            edge_index=edge_index,      # [2,E]
            y=label,                    # scalar
            num_nodes=num_nodes,
            central_mask=torch.zeros(num_nodes, dtype=torch.bool).index_fill_(0, torch.tensor([0]), True)
        )


In [74]:
# ---------------------------------------------------------------------------
# 3. Model
# ---------------------------------------------------------------------------

class RetweetGNN(nn.Module):
    """
    Architecture:
        1. Pretrained embedding lookup  (frozen)
        2. Feed-forward projection      (trainable, adds retweet_flag)
        3. MagNetConv layers            (directed message passing via magnetic Laplacian)
        4. TransformerConv layer        (attention-based aggregation)
        5. Readout: central node repr   (not global pool — we care about node 0)
        6. MLP classifier + softmax
    """

    def __init__(
        self,
        embeddings_path: str,
        device: str,
        ff_hidden_dim: int = 256,
        gcn_hidden_dim: int = 128,
        transformer_dim: int = 128,
        transformer_heads: int = 4,
        num_classes: int = 2,
        dropout: float = 0.3,
        q: float = 0.25,    # MagNet phase parameter: controls directional sensitivity
        K: int = 1,         # Chebyshev order for MagNetConv
    ):
        super().__init__()

        # 1. Frozen pretrained embeddings
        self.lookup = PretrainedEmbeddingLookup(embeddings_path, device)
        embed_dim = self.lookup.embedding_dim  # e.g. 128 from MagNet

        # Input to FF: embedding + retweet_flag (1 dim)
        ff_input_dim = embed_dim + 1

        # 2. Feed-forward projection (makes embeddings trainable/adaptable)
        self.ff = nn.Sequential(
            nn.Linear(ff_input_dim, ff_hidden_dim),
            nn.LayerNorm(ff_hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(ff_hidden_dim, gcn_hidden_dim),
            nn.LayerNorm(gcn_hidden_dim),
            nn.GELU(),
        )

        # 3. MagNetConv layers
        # Each layer takes (x_real, x_imag) and returns (x_real, x_imag).
        # The imaginary stream carries directional phase information throughout.
        self.magnet1 = MagNetConv(gcn_hidden_dim, gcn_hidden_dim, q=q, K=K, trainable_q=True)
        self.magnet2 = MagNetConv(gcn_hidden_dim, gcn_hidden_dim, q=q, K=K, trainable_q=True)

        # 4. Transformer layer
        self.transformer = TransformerConv(
            in_channels=gcn_hidden_dim * 2,  # real + imag concatenated
            out_channels=transformer_dim // transformer_heads,
            heads=transformer_heads,
            dropout=dropout,
            concat=True,   # output dim = transformer_dim
        )

        # 5. Classifier (applied to central node only)
        self.classifier = nn.Sequential(
            nn.Linear(transformer_dim, transformer_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(transformer_dim // 2, num_classes),
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, data):
        user_ids     = data.user_ids       # [N_total]
        retweet_flag = data.retweet_flag   # [N_total, 1]
        edge_index   = data.edge_index     # [2, E_total]
        batch        = data.batch          # [N_total] — PyG batch vector
        central_mask = data.central_mask   # [N_total] bool, True for node 0 per graph

        # 1. Lookup pretrained embeddings (no grad)
        with torch.no_grad():
            pretrained = self.lookup(user_ids)  # [N, embed_dim]

        # 2. Concatenate retweet flag and project
        x = torch.cat([pretrained, retweet_flag], dim=-1)  # [N, embed_dim+1]
        x = self.ff(x)                                      # [N, gcn_hidden_dim]

        # 3. MagNetConv layers
        # MagNetConv signature: forward(x_real, x_imag, edge_index) -> (x_real, x_imag)
        # We start with x as the real part and zeros as the imaginary part.
        # The imaginary stream accumulates directional phase information across layers.
        x_real, x_imag = x, torch.zeros_like(x)

        x_real_res, x_imag_res = x_real, x_imag
        x_real, x_imag = self.magnet1(x_real, x_imag, edge_index)
        x_real, x_imag = F.gelu(x_real), F.gelu(x_imag)
        x_real, x_imag = self.dropout(x_real), self.dropout(x_imag)

        x_real, x_imag = self.magnet2(x_real, x_imag, edge_index)
        x_real = x_real + x_real_res   # residual on real stream
        x_imag = x_imag + x_imag_res   # residual on imaginary stream
        x_real, x_imag = F.gelu(x_real), F.gelu(x_imag)

        # Merge real and imaginary into a single representation before TransformerConv.
        # Concatenation preserves both undirected structure (real) and
        # directional phase information (imaginary) for the attention layer.
        x = torch.cat([x_real, x_imag], dim=-1)  # [N, 2*gcn_hidden_dim]

        # 4. Transformer layer
        x = self.transformer(x, edge_index)                 # [N, transformer_dim]
        x = F.gelu(x)

        # 5. Extract central node representation per graph in the batch
        central_x = x[central_mask]                         # [batch_size, transformer_dim]

        # 6. Classify
        logits = self.classifier(central_x)                 # [batch_size, 2]
        return logits  # raw logits; use cross_entropy in training


In [75]:
# ---------------------------------------------------------------------------
# 4. Training & Evaluation
# ---------------------------------------------------------------------------

def train_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0
    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        logits = model(batch)
        loss = F.cross_entropy(logits, batch.y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    correct = total = 0
    all_probs, all_labels = [], []
    for batch in loader:
        batch = batch.to(device)
        logits = model(batch)
        probs = F.softmax(logits, dim=-1)[:, 1]
        preds = logits.argmax(dim=-1)
        correct += (preds == batch.y).sum().item()
        total += batch.y.size(0)
        all_probs.append(probs.cpu())
        all_labels.append(batch.y.cpu())
    acc = correct / total
    all_probs  = torch.cat(all_probs)
    all_labels = torch.cat(all_labels)
    return acc, all_probs, all_labels


def build_and_train(
    raw_train_samples: list,
    raw_val_samples: list,
    embeddings_path: str,
    epochs: int = 50,
    batch_size: int = 32,
    lr: float = 1e-3,
    device: str = "cuda" if torch.cuda.is_available() else "cpu",
):
    train_ds = RetweetDataset(raw_train_samples)
    val_ds   = RetweetDataset(raw_val_samples)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False)

    model = RetweetGNN(embeddings_path=embeddings_path, device=device).to(device)

    # Only optimize non-frozen parameters
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr, weight_decay=1e-4
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    print(f"Training on {device} | {len(train_ds)} train / {len(val_ds)} val samples")
    print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

    best_val_acc = 0
    for epoch in range(1, epochs + 1):
        loss = train_epoch(model, train_loader, optimizer, device)
        val_acc, _, _ = evaluate(model, val_loader, device)
        scheduler.step()

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), "best_retweet_gnn.pt")

        if epoch % 5 == 0 or True:
            print(f"Epoch {epoch:>3} | Loss: {loss:.4f} | Val Acc: {val_acc:.4f} | Best: {best_val_acc:.4f}")

    print(f"\nTraining complete. Best val accuracy: {best_val_acc:.4f}")
    return model


In [76]:

# ---------------------------------------------------------------------------
# 5. Usage example
# ---------------------------------------------------------------------------

# if __name__ == "__main__":
# Example: construct a single sample manually
sample = {
    "central_user_id": 42,
    "neighbor_ids": [7, 15, 99, 204, 301],
    "retweeted_ids": [15, 99],          # these neighbors retweeted
    "edge_index": [                     # LOCAL indices (0=central, 1..5=neighbors)
        (0, 1), (1, 0),
        (0, 2), (2, 0),
        (1, 3), (3, 4),
        (0, 5), (5, 0),
    ],
    "label": 1,                         # central user DID retweet
}

train_samples = [sample] * 1000

val_samples = train_samples[:100]

# Build dataset from your list of samples and train:
model = build_and_train(
    raw_train_samples=train_samples,
    raw_val_samples=val_samples,
    embeddings_path="node_embeddings.pt",
    epochs=10,
    device=device
)

Training on cuda:1 | 1000 train / 100 val samples
Trainable params: 272,708
Epoch   1 | Loss: 0.0350 | Val Acc: 1.0000 | Best: 1.0000
Epoch   2 | Loss: 0.0000 | Val Acc: 1.0000 | Best: 1.0000
Epoch   3 | Loss: 0.0000 | Val Acc: 1.0000 | Best: 1.0000
Epoch   4 | Loss: 0.0000 | Val Acc: 1.0000 | Best: 1.0000
Epoch   5 | Loss: 0.0000 | Val Acc: 1.0000 | Best: 1.0000
Epoch   6 | Loss: 0.0000 | Val Acc: 1.0000 | Best: 1.0000
Epoch   7 | Loss: 0.0000 | Val Acc: 1.0000 | Best: 1.0000
Epoch   8 | Loss: 0.0000 | Val Acc: 1.0000 | Best: 1.0000
Epoch   9 | Loss: 0.0000 | Val Acc: 1.0000 | Best: 1.0000
Epoch  10 | Loss: 0.0000 | Val Acc: 1.0000 | Best: 1.0000

Training complete. Best val accuracy: 1.0000


- Load one user and transform to data to the input format of the GNN model

In [77]:
import json
from random import sample

with open("../../data/datasets/user_splits.json") as f:
    user_splits = json.load(f)
user_ids = sample(user_splits["u_train"], 10)
uid = user_ids[0]
uid

283762641

In [78]:
from utils import load_dataframe_raw
data_uid = load_dataframe_raw(uid, sparse=True)
X_tr, X_te, y_tr, y_te = data_uid

Processing 283762641
Path exists, trying to load
OK


In [79]:
from utils import create_gnn_train_val_samples
from tw_dataset.settings import IG_GRAPH_PATH
import networkx as nx

graph = nx.read_graphml(IG_GRAPH_PATH)
central_user_id = uid
train_gnn_samples, val_gnn_samples = create_gnn_train_val_samples(central_user_id, graph, X_tr, y_tr, X_te, y_te)

In [80]:
len(train_gnn_samples)

3500

In [81]:
len(val_gnn_samples)

1500

In [ ]:
# Build dataset from your list of samples and train:
model = build_and_train(
    raw_train_samples=train_gnn_samples[:350],
    raw_val_samples=val_gnn_samples[:150],
    embeddings_path="node_embeddings.pt",
    epochs=10,
    device=device
)

Training on cuda:1 | 350 train / 150 val samples
Trainable params: 272,708
Epoch   1 | Loss: 0.5029 | Val Acc: 0.8733 | Best: 0.8733
Epoch   2 | Loss: 0.4418 | Val Acc: 0.8733 | Best: 0.8733
Epoch   3 | Loss: 0.4417 | Val Acc: 0.8733 | Best: 0.8733
Epoch   4 | Loss: 0.4416 | Val Acc: 0.8733 | Best: 0.8733
Epoch   5 | Loss: 0.4394 | Val Acc: 0.8733 | Best: 0.8733
